# Scan Noise Inspection Notebook

This notebook is a manual workflow for checking PolyDocBench scan noising.

Use it to:

1. Render a clean PDF + GT pair if needed.
2. Generate noisy scan variants with the project noise profiles.
3. Generate paired transformed GT for noisy images.
4. Inspect affine transforms and coordinate systems.
5. Draw GT overlays to visually verify the noisy-image annotations.

The notebook uses the public `polydocbench.noise` module. It does not duplicate the noising implementation.

## 1. Environment Setup

Run this notebook from the repository root or from the `notebooks/` directory. The cell below detects the project root and imports project modules.

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import sys
from pathlib import Path

from PIL import Image

try:
    from IPython.display import Image as IPyImage
    from IPython.display import Markdown, display
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value

    class IPyImage:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return self.filename

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 2. Noise Profiles

PolyDocBench currently includes three scan noise profiles:

| Profile | Main operations | Typical use |
| --- | --- | --- |
| `light_scan` | resolution reduction, illumination gradient, JPEG artifacts | mild scan-like perturbation |
| `medium_scan` | `light_scan` + affine rotation/shear | rotated or slightly skewed scans |
| `heavy_scan` | `medium_scan` + ink morphology | stronger document image artifacts |

The exact random values are controlled by `SEED`.

In [ ]:
from polydocbench.noise import NOISE_PROFILES

for profile_name, pipeline in NOISE_PROFILES.items():
    operations = [fn.__name__ for fn in pipeline]
    print(f"{profile_name}: {', '.join(operations)}")

## 3. Experiment Configuration

Change these paths and settings to inspect another document or profile.

If `RENDER_INPUT_IF_NEEDED = True`, the notebook renders `SOURCE_JSON` into `PDF_PATH` and `GT_PATH` before generating noisy scans.

In [ ]:
SOURCE_JSON = PROJECT_ROOT / "examples" / "wiki_formulas.json"
PDF_PATH = PROJECT_ROOT / "outputs" / "notebook_noise" / "wiki_formulas.pdf"
GT_PATH = PROJECT_ROOT / "outputs" / "notebook_noise" / "wiki_formulas_gt.json"
NOISY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_noise" / "noisy"
DEBUG_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_noise" / "debug"

TEMPLATE = "simple_article"
FONT_PATH = PROJECT_ROOT / "DejaVu Sans" / "DejaVuSans.ttf"
RENDER_INPUT_IF_NEEDED = True

PAGE_INDEX = 0
DPI = 200
SEED = 42
VARIANTS = 1
SELECTED_PROFILES = ["light_scan", "medium_scan", "heavy_scan"]

print("Source JSON:", SOURCE_JSON)
print("PDF:", PDF_PATH)
print("GT:", GT_PATH)
print("Noisy output:", NOISY_OUTPUT_DIR)
print("Profiles:", SELECTED_PROFILES)

## 4. Render Clean PDF And GT

This step creates the clean input document used by the noising pipeline. Existing files are reused unless `RENDER_INPUT_IF_NEEDED` is enabled and one of the files is missing.

In [ ]:
from polydocbench.layout import LayoutEngine
from polydocbench.render import PDFRenderer

PDF_PATH.parent.mkdir(parents=True, exist_ok=True)

if RENDER_INPUT_IF_NEEDED and (not PDF_PATH.exists() or not GT_PATH.exists()):
    with contextlib.redirect_stdout(io.StringIO()):
        layout = LayoutEngine(font_path=FONT_PATH if FONT_PATH.exists() else None).layout_document(
            SOURCE_JSON,
            template_name=TEMPLATE,
        )
        result = PDFRenderer(debug=False).render(layout, PDF_PATH)
    print("Rendered PDF:", result["pdf_path"])
    print("Rendered GT:", result["gt_path"])
    print("Pages:", len(layout.pages))
    print("Elements:", len(layout.elements))
else:
    print("Using existing PDF and GT")

assert PDF_PATH.exists(), f"PDF was not found: {PDF_PATH}"
assert GT_PATH.exists(), f"GT was not found: {GT_PATH}"

## 5. Generate Noisy Scans Without GT

Use this when you only need images and do not need transformed annotations.

In [ ]:
from polydocbench.noise import pdf_to_noisy_images

images_only_dir = NOISY_OUTPUT_DIR / "images_only"
images_only_result = pdf_to_noisy_images(
    pdf_path=PDF_PATH,
    output_dir=images_only_dir,
    page_index=PAGE_INDEX,
    n_variants=VARIANTS,
    seed=SEED,
    dpi=DPI,
    profiles=SELECTED_PROFILES,
)

print("Zoom:", images_only_result["zoom"])
for image_path in images_only_result["images"]:
    print(image_path)

## 6. Generate Noisy Scans With Transformed GT

This is the recommended mode for OCR and layout evaluation. It writes one image and one GT file per profile/variant.

For `medium_scan` and `heavy_scan`, the GT stores transformed polygons and bboxes in image pixel coordinates.

In [ ]:
from polydocbench.noise import pdf_to_noisy_dataset

paired_result = pdf_to_noisy_dataset(
    pdf_path=PDF_PATH,
    gt_path=GT_PATH,
    output_dir=NOISY_OUTPUT_DIR,
    page_index=PAGE_INDEX,
    n_variants=VARIANTS,
    seed=SEED,
    dpi=DPI,
    profiles=SELECTED_PROFILES,
)

print("Zoom:", paired_result["zoom"])
print("Artifacts:")
for artifact in paired_result["artifacts"]:
    print(json.dumps(artifact, indent=2))

## 7. Inspect One Noisy Artifact

Choose a profile and variant to inspect. Start with `medium_scan` when you want to check rotation-aware GT.

In [ ]:
PROFILE_TO_INSPECT = "heavy_scan"
VARIANT_TO_INSPECT = 0

IMAGE_PATH = NOISY_OUTPUT_DIR / f"{PROFILE_TO_INSPECT}_{VARIANT_TO_INSPECT}.jpg"
NOISY_GT_PATH = NOISY_OUTPUT_DIR / f"{PROFILE_TO_INSPECT}_{VARIANT_TO_INSPECT}_gt.json"

assert IMAGE_PATH.exists(), f"Noisy image was not found: {IMAGE_PATH}"
assert NOISY_GT_PATH.exists(), f"Noisy GT was not found: {NOISY_GT_PATH}"

image = Image.open(IMAGE_PATH).convert("RGB")
noisy_gt = json.loads(NOISY_GT_PATH.read_text(encoding="utf-8"))
metadata = noisy_gt.get("metadata", {})
coordinate_system = metadata.get("coordinate_system", {})
transform = metadata.get("transform", {})

display(Markdown(f"**Image:** `{IMAGE_PATH}`"))
display(Markdown(f"**Image size:** {image.width} x {image.height}"))
display(Markdown(f"**GT:** `{NOISY_GT_PATH}`"))
display(Markdown(f"**Coordinate system:** `{coordinate_system}`"))
display(Markdown(f"**Transform:** `{transform}`"))

display(IPyImage(filename=str(IMAGE_PATH)))

## 8. Sanity Checks

These checks verify that the paired GT is ready for OCR/layout evaluation.

In [ ]:
def iter_gt_elements(gt_json: dict) -> list[dict]:
    elements = []
    for page in gt_json.get("pages", []):
        for container in page.get("containers", []):
            elements.extend(container.get("elements", []))
    return elements or list(gt_json.get("elements", []))


elements = iter_gt_elements(noisy_gt)
polygon_count = sum(1 for element in elements if element.get("polygon"))
source_bbox_count = sum(1 for element in elements if element.get("metadata", {}).get("source_bbox"))
identity_matrix = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]
matrix = transform.get("matrix", [])
non_identity_transform = matrix != identity_matrix

print("Elements:", len(elements))
print("Elements with polygon:", polygon_count)
print("Elements with source_bbox:", source_bbox_count)
print("Non-identity transform:", non_identity_transform)

assert coordinate_system.get("unit") == "pixels", "Expected pixel coordinates"
assert coordinate_system.get("origin") == "top-left", "Expected top-left image origin"
assert polygon_count == len(elements), "Every element should have transformed polygon geometry"
assert source_bbox_count == len(elements), "Every element should preserve source_bbox metadata"

if PROFILE_TO_INSPECT in {"medium_scan", "heavy_scan"}:
    assert non_identity_transform, "Expected a non-identity transform for affine profile"

## 9. Draw GT Overlays

Color convention:

- Red: transformed GT polygon.
- Blue: axis-aligned GT bbox around the polygon.

For rotated scans, red polygons are the most important visual check.

In [ ]:
from polydocbench.noise import draw_gt_overlay

DEBUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
polygon_overlay = DEBUG_OUTPUT_DIR / f"{PROFILE_TO_INSPECT}_{VARIANT_TO_INSPECT}_polygon.jpg"
bbox_overlay = DEBUG_OUTPUT_DIR / f"{PROFILE_TO_INSPECT}_{VARIANT_TO_INSPECT}_bbox.jpg"
both_overlay = DEBUG_OUTPUT_DIR / f"{PROFILE_TO_INSPECT}_{VARIANT_TO_INSPECT}_both.jpg"

draw_gt_overlay(IMAGE_PATH, NOISY_GT_PATH, polygon_overlay, mode="polygon", polygon_color="red", line_width=2)
draw_gt_overlay(IMAGE_PATH, NOISY_GT_PATH, bbox_overlay, mode="bbox", bbox_color="blue", line_width=2)
draw_gt_overlay(IMAGE_PATH, NOISY_GT_PATH, both_overlay, mode="both", polygon_color="red", bbox_color="blue", line_width=2)

print("Polygon overlay:", polygon_overlay)
print("BBox overlay:", bbox_overlay)
print("Combined overlay:", both_overlay)

display(IPyImage(filename=str(polygon_overlay)))

## 10. Compare Profiles Side By Side

This cell shows the first generated variant for each selected profile.

In [ ]:
for profile_name in SELECTED_PROFILES:
    path = NOISY_OUTPUT_DIR / f"{profile_name}_0.jpg"
    if path.exists():
        display(Markdown(f"### {profile_name}"))
        display(IPyImage(filename=str(path)))